# HBCC — CIFAR-10 | Version 1: No Augmentation, CE Only

**Purpose:** Ablation that isolates the HBCC architecture advantage from the training recipe.  
All models — ResNet-18, MobileNetV2, ShuffleNetV2, CoC-baseline, HBCC-Small, HBCC-Medium —  
are trained with **zero data augmentation** (no RandAugment, no MixUp, no CutMix, no Random Erasing).  
HBCC uses **CE loss only** (no Knowledge Distillation).

**Scientific question answered:** Does HBCC's hybrid architecture outperform baselines when the  
training recipe is identical and stripped to its minimum?

**Timing estimate (single 12 h Kaggle GPU session):**
- ResNet-18 200 ep, no aug: ~1.5 h
- MobileNetV2 200 ep, no aug: ~0.3 h
- ShuffleNetV2 200 ep, no aug: ~0.3 h
- CoC-baseline 200 ep, no aug: ~0.7 h
- HBCC-Medium 250 ep, no aug: ~3.4 h
- HBCC-Small 250 ep, no aug: ~3.1 h
- Benchmarks: ~0.4 h
- **Total: ~9.7 h** (comfortable 2+ h buffer within 12 h limit)

Results are saved to `results/cifar10_noaug_ce_results.csv`.

In [ ]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster
!pip install fvcore -q

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run from repo root or notebooks dir. Got {ROOT}"

PYTHON = Path(sys.executable)
print("Repo:", ROOT)
print("Python:", PYTHON)


def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def train(
    config: str,
    output: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args([*(overrides or []), f"train.seed={SEED}"])
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    checkpoint: str | None = None,
    experiment_name: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs   = DEBUG_BENCHMARK_RUNS   if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py", "--config", config,
        "--output", BENCHMARK_OUTPUT,
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup), "--runs", str(runs),
    ]
    if experiment_name:
        args += override_args([f"experiment.name={experiment_name}"])
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"  [skip] checkpoint not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)

## Phase Switches

All phases are enabled by default for a single-session run.  
Set a flag to `False` to skip a phase (e.g. if its checkpoint already exists from a previous run).

In [ ]:
RUN_ENV_CHECKS          = True
RUN_TEACHER             = True   # ResNet-18 (no aug) — also used as accuracy reference
RUN_LIGHTWEIGHT_BASELINES = True  # MobileNetV2 + ShuffleNetV2 (no aug)
RUN_COC_BASELINE        = True   # CoC-baseline (no aug)
RUN_STUDENT_CE          = True   # HBCC-Small + HBCC-Medium CE only (no aug, no KD)
RUN_BENCHMARKS          = True

# ---------------------------------------------------------------------------
# Core augmentation-disable overrides — applied to EVERY training run.
# data.augment=false  → disables RandomCrop, RandomHFlip, RandAugment, RandomErasing
# mixup/cutmix = 0    → disables mix-based augmentation (handled in train_one_epoch)
# ---------------------------------------------------------------------------
NO_AUG_OVERRIDES = [
    "data.augment=false",
    "train.mixup_alpha=0.0",
    "train.cutmix_alpha=0.0",
    "train.cutmix_prob=0.0",
]

# Epoch budgets
# Without augmentation models converge faster → 200 ep for baselines is sufficient.
# HBCC uses 250 ep to ensure the more complex model is fully converged.
FULL_EPOCHS     = 200   # baselines: ResNet-18, MobileNetV2, ShuffleNetV2, CoC
ACCURACY_EPOCHS = 250   # HBCC-Small, HBCC-Medium

SEED = 42

# Benchmark settings
BENCHMARK_OUTPUT        = "results/benchmark_noaug_ce"
BENCHMARK_BATCHES       = [1, 16, 64, 128]
BENCHMARK_WARMUP        = 30
BENCHMARK_RUNS          = 100
DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP  = 2
DEBUG_BENCHMARK_RUNS    = 3

# Output directories — separate from the current-version runs to avoid conflicts
DIR_TEACHER   = "runs_noaug_teacher"
DIR_BASELINES = "runs_noaug_baselines"
DIR_COC       = "runs_noaug_coc"
DIR_ACCURACY  = "runs_noaug_accuracy"

# Experiment names (used to label checkpoints and benchmark JSON files)
EXP_RESNET     = "resnet18_noaug"
EXP_MOBILENET  = "mobilenet_v2_noaug"
EXP_SHUFFLENET = "shufflenet_v2_x1_0_noaug"
EXP_COC        = "coc_baseline_noaug"
EXP_HBCC_SML   = "hbcc_accuracy_small_noaug"
EXP_HBCC_MED   = "hbcc_accuracy_medium_noaug"

print("NO_AUG_OVERRIDES:", NO_AUG_OVERRIDES)

## 0. Environment Checks

In [ ]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_accuracy_medium.yaml", "--batch-size", "1"])

## 1. Teacher / Reference Baseline — ResNet-18 (no augmentation)

ResNet-18 is trained from scratch with **no augmentation**.  
It serves as the upper-bound accuracy reference in the no-aug comparison table.

In [ ]:
if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER,
        overrides=[
            f"experiment.name={EXP_RESNET}",
            f"train.epochs={FULL_EPOCHS}",
            *NO_AUG_OVERRIDES,
        ],
    )

## 2. Lightweight Baselines — MobileNetV2 & ShuffleNetV2 (no augmentation)

In [ ]:
if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        DIR_BASELINES,
        overrides=[
            f"experiment.name={EXP_MOBILENET}",
            f"train.epochs={FULL_EPOCHS}",
            *NO_AUG_OVERRIDES,
        ],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        DIR_BASELINES,
        overrides=[
            f"experiment.name={EXP_SHUFFLENET}",
            f"train.epochs={FULL_EPOCHS}",
            *NO_AUG_OVERRIDES,
        ],
    )

## 3. CoC Baseline (no augmentation)

In [ ]:
COC_CHECKPOINT = f"{DIR_COC}/{EXP_COC}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        DIR_COC,
        overrides=[
            f"experiment.name={EXP_COC}",
            f"train.epochs={FULL_EPOCHS}",
            *NO_AUG_OVERRIDES,
        ],
    )

## 4. HBCC CE Training (no augmentation, no KD)

Both HBCC variants are trained with cross-entropy loss only.  
No teacher, no knowledge distillation. This isolates the architectural contribution.

In [ ]:
if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_ACCURACY,
        overrides=[
            f"experiment.name={EXP_HBCC_SML}",
            f"train.epochs={ACCURACY_EPOCHS}",
            *NO_AUG_OVERRIDES,
        ],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_ACCURACY,
        overrides=[
            f"experiment.name={EXP_HBCC_MED}",
            f"train.epochs={ACCURACY_EPOCHS}",
            *NO_AUG_OVERRIDES,
        ],
    )

## 5. Benchmark Matrix

Latency, params, and MACs are architecture properties — identical regardless of training recipe.  
This step measures them against the no-aug checkpoints to confirm accuracy alongside hardware metrics.

In [ ]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml",           EXP_RESNET,     f"{DIR_TEACHER}/{EXP_RESNET}/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml",       EXP_MOBILENET,  f"{DIR_BASELINES}/{EXP_MOBILENET}/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml",EXP_SHUFFLENET, f"{DIR_BASELINES}/{EXP_SHUFFLENET}/best.pth"),
    ("configs/coc_cifar_baseline.yaml",                 EXP_COC,        COC_CHECKPOINT),
    ("configs/hbcc_accuracy_small.yaml",                EXP_HBCC_SML,   f"{DIR_ACCURACY}/{EXP_HBCC_SML}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",               EXP_HBCC_MED,   f"{DIR_ACCURACY}/{EXP_HBCC_MED}/best.pth"),
]

if RUN_BENCHMARKS:
    for cfg, exp_name, ckpt in BENCHMARK_JOBS:
        benchmark(cfg, checkpoint=ckpt, experiment_name=exp_name)

## 6. Collect Metrics and Build Results Table

In [ ]:
NOAUG_RUN_DIRS = [DIR_TEACHER, DIR_BASELINES, DIR_COC, DIR_ACCURACY]


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for run_root in NOAUG_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record = (
                next((r for r in reversed(test_records) if r.get("epoch") == best.get("epoch")), test_records[-1])
                if test_records else {}
            )
            rows.append({
                "experiment": metrics_path.parent.name,
                "best_epoch": best.get("epoch"),
                "best_val_acc1": best.get("val_acc1"),
                "test_acc1": test_record.get("test_acc1"),
            })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


train_df = collect_train_metrics()
bench_df = collect_benchmarks()
print("Training results:")
display(train_df)

In [ ]:
def build_publication_table(bench: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return pd.DataFrame()
    out = bench.copy()
    acc_map = {}
    if not train.empty:
        acc_source = "test_acc1" if "test_acc1" in train.columns else "best_val_acc1"
        acc_map = dict(zip(train["experiment"], train[acc_source].fillna(train["best_val_acc1"])))
    out["acc1"] = out["config_id"].map(acc_map)
    rename = {
        "config_id": "Model",
        "acc1": "Top-1 Acc (%)",
        "params_total": "Params",
        "macs": "MACs",
        "latency_ms_b1": "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    cols = [c for c in rename if c in out.columns]
    table = out[cols].rename(columns=rename).dropna(subset=["Top-1 Acc (%)"]).sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)

pub_df = build_publication_table(bench_df, train_df)
pub_df.to_csv(out_dir / "cifar10_noaug_ce_results.csv", index=False)
print("\n=== CIFAR-10 | No Augmentation | CE Only ===")
display(pub_df)